
<div style="text-align: right;">
  <a href="../../Files/summarize_medical_record_proxy.zip"
     download="summarize_medical_record_proxy.zip"
     class="md-button md-button--primary">
    ⬇️ Download Notebook
  </a>


## Install and import utility.ipynb

In [1]:
%run ./utility.ipynb

In [ ]:
%pip install -e ./utility.ipynb
%pip install import_ipynb

In [3]:
import import_ipynb
import utility

## Build prompt

In [4]:
def build_prompt(textract_response):
    prompt = """Extracted text from medical records (via Textract) is provided below. Summarize it into a concise, structured patient history including:
                    • Diagnoses
                    • Medications (name, dose, frequency)
                    • Allergies
                    • Procedures
                    • Relevant labs
                    • Timeline of key events
                    Output in clean JSON. Exclude irrelevant text, don’t infer missing info, and mark sections with "None reported" if not found.
                    Input:
                    textract_output_text:{textract_response}""".format(textract_response=textract_response)
    return prompt

## Build request body

In [5]:
def build_textract_request(pdf_path):
    base64encoded = utility.base64_encode_file(pdf_path=pdf_path)
    textract_request_body={
        "methodName" : "start_document_analysis",
        "parameters" : {
            "featureTypes" : ["layout"],
            "document" : {
                "type" : "pdf",
                "base64EncodedData" : base64encoded
            }
        }
    }
    return textract_request_body

def build_bedrock_request(prompt):
    #payload for converse route, claude model
    inferenceConfig = {
            "maxTokens": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
    body = {
        "inferenceConfig": inferenceConfig,
        "messages": [
            {   "role": "user",
                "content": [
                    {"text": prompt}]
            }]
        }
    bedrock_request_body={
        "methodName": "converse",
        "parameters" : { "body" : body }
    }
    return bedrock_request_body

## Sending the Medical Record PDF to Textract and passing the parsed output to Bedrock

In [ ]:
def invoke_modelops(model_id, textract_request_body) -> Dict:
    token = utility.get_oauth_token()
    try:
        #invoke textract
        print("invoked textract")
        ocr_jobId = utility.invoke_ocr_proxy(token=token, request_body = textract_request_body)
        if utility.wait_for_job_completion(ocr_jobId, token):
            textract_resp = utility.get_response(ocr_jobId, token, output_type='text')
            print("recieved text response")
        else:
            textract_resp = None
        # invoke bedrock
        prompt = build_prompt(textract_resp)
        claude_request = build_bedrock_request(prompt)
        print("invoked bedrock")
        llm_jobId = utility.invoke_llm_proxy(model_id=model_id, token=token, request_body=claude_request)
        if utility.wait_for_job_completion(llm_jobId, token):
            bedrock_resp = utility.get_response(llm_jobId, token)
            final_response={}
            final_response["body"] = json.dumps(bedrock_resp["output"]['message']['content'][0]['text'])
        else:
            final_response = {}

        return final_response
    except Exception as e:
        raise Exception(f"Issue in invoking modelOps: {e}")

## Invocation

In [ ]:
modelId = "anthropic.claude-3-5-sonnet-20241022-v2:0"
textract_request = build_textract_request('D:/Users/572981/Downloads/example_usecase_refactored/medical_report_sample.pdf')
response = invoke_modelops(model_id=modelId,textract_request_body=textract_request)
print(response)